# Sector-scale blind run — PS7 objectives O2–O6
### AI-enabled detection of exoplanets from a real TESS sector

This is the deliverable that matches the problem statement directly: take **one TESS sector's 2-minute light curves** (downloaded in bulk from MAST), run the pipeline **blind** across the slice, then train a classifier on the *known* targets and apply it to the *unknown* ones.

**Dataset note.** The PS link `archive.stsci.edu/tess/tic_ctl.html` is the TIC/CTL **star catalog** (metadata only — no photometry). The *light curves* come from the per-sector bulk script `tesscurl_sector_NN_lc.sh`. We use **Sector 5** because it contains our validated anchors TOI 700 (TIC 307210830) and TOI-270 (TIC 259377017).

| Objective | Where |
|---|---|
| **O2** identify the event | blind BLS in `scan.scan_target` |
| **O3** characterize + shape params | `scan.characterize_top` (MCMC) |
| **O4** train classifier on known data | `classify.train` on TOI labels |
| **O5** apply to unknown, classify | `classify.predict` over the slice |
| **O6** params + significance + robustness | MCMC ± errors, SDE/SNR/FAP, injection |


In [ ]:
# --- Setup -------------------------------------------------------------------------
# LOCAL: imports the `exopipeline` package from the project root.
# COLAB: upload the `exopipeline/` folder next to this notebook, then run the install cell.
import sys, os
for p in [".", "..", r"g:\Exoplanet project"]:
    if os.path.isdir(os.path.join(p, "exopipeline")):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from exopipeline import ingest, scan, classify, labels, config
SECTOR = config.DEFAULT_SECTOR
print("exopipeline ready | sector =", SECTOR)


## 1. The dataset — one sector's 2-min light curves
Parse the sector's bulk-download manifest (no FITS yet). Each row is one star; the FITS is fetched on demand during the scan. ~20k targets per sector.

In [ ]:
manifest = ingest.sector_lc_manifest(SECTOR)
print(f"Sector {SECTOR}: {len(manifest):,} two-minute targets in the bulk manifest")
manifest.head(5)

## 2. Blind scan of the slice (O2 detect · O5 classify)
We process a representative **unbiased slice** (the first `SLICE_SIZE` stars + every known TOI in the sector, for validation). Each star runs the *cheap* tier: detrend → blind BLS → vetting features → classify (type) → significance (SDE/SNR/FAP). Results are checkpointed to CSV so the run is resumable.

> Tune `n` (slice size) and `limit` (cap per run) to taste. A full ~4k slice takes a couple of hours; start with a few hundred to see the workflow.

In [ ]:
# Set limit=None and n=config.SLICE_SIZE for the full submission run.
# keep_fits=False streams (deletes each FITS after loading) to cap disk use.
cands = scan.scan_slice(sector=SECTOR, n=config.SLICE_SIZE,
                        limit=300,          # <-- raise for the real run
                        keep_fits=True, verbose=True)
ok = cands[cands["status"] == "ok"]
print(f"\nprocessed {len(cands)} | detections {len(ok)} | "
      f"known TOIs in slice {int(cands['is_known_toi'].sum())}")

### O2/O3 — what did the blind search find?
Distribution of recovered periods and detection significance across the slice.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(ok["period"].dropna(), bins=30, color="#3b7dd8")
ax[0].set(xlabel="recovered period (d)", ylabel="# stars", title="O2: detected periods")
ax[1].hist(ok["sde"].dropna(), bins=30, color="#d8763b")
ax[1].axvline(config.SDE_THRESHOLD, color="k", ls="--", label=f"SDE={config.SDE_THRESHOLD}")
ax[1].set(xlabel="BLS SDE", ylabel="# stars", title="O6: significance"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Train the classifier on the known dataset (O4)
The classifier is trained on the **broad TOI labelled set (hundreds of stars across the catalog)** — not just this sector — so it generalises. We build (or load) the feature table, then train + isotonically calibrate a LightGBM and report the confusion matrix / PR-AUC. (Training breadth = accuracy; the sector's own TOIs are held out below as a validation check.)

In [ ]:
# Build the feature table once (downloads a few hundred labelled stars), then cache.
if config.FEATURE_TABLE.exists():
    feat = classify.load_feature_table()
else:
    toi = labels.fetch_toi()
    sample = labels.balanced_sample(toi, per_class=60)
    feat = labels.build_feature_table(sample)      # slow: one-time download+compute
print("feature table:", feat.shape)
res = classify.train(feat)
classify.save_model(res["model"])
print("PR-AUC (macro):", round(res["pr_auc"], 3))
print(pd.DataFrame(res["confusion_matrix"], index=res["classes"], columns=res["classes"]))

## 4. Apply to the unknowns & validate on the knowns (O5)
Re-score the slice with the trained model, then (a) rank the **unknown** stars as planet candidates, and (b) check predictions against the sector's **known** TOI dispositions as a validation.

In [ ]:
model = classify.load_model()
# unknown candidates, ranked by significance
unknown = ok[~ok["is_known_toi"]].copy()
ranked = scan.rank_candidates(unknown)
print("Top blind planet candidates among UNKNOWN stars:")
print(ranked[["tic","period","depth_ppm","sde","snr","pred_class","confidence"]]
      .head(10).to_string(index=False))
# validation on KNOWN TOIs present in the slice
known = ok[ok["is_known_toi"]]
if len(known):
    vt = known.dropna(subset=["true_label"])
    acc = (vt["pred_class"] == vt["true_label"]).mean() if len(vt) else float("nan")
    print(f"\nValidation on {len(vt)} known TOIs in slice: accuracy = {acc:.2f}")

## 5. Characterize the top events (O3 shape · O6 parameters)
The *expensive* tier runs only on the detected events: full `batman`+`emcee` shape fit (depth, duration, Rp, impact parameter, U-vs-V) with posterior credible intervals, plus the one-page vetting sheet. Here we characterise the top few candidates from the slice.

In [ ]:
# Supply catalog stellar params for the anchors -> density prior breaks the
# grazing degeneracy for shallow transits.
stellar = {259377017: (0.378, 0.386)}   # TOI-270 R*, M* (Sun units)
summary = scan.characterize_top(ok, k=2, stellar=stellar, nsteps=1500, nburn=500)
for s in summary:
    print(f"{s['tic']}: {s['verdict']} ({s['confidence']*100:.0f}%)  "
          f"Rp={s['Rp']:.2f} Re  P={s['period']:.4f} d  -> {s['sheet']}")

In [ ]:
from PIL import Image
if summary:
    img = Image.open(summary[0]["sheet"])
    plt.figure(figsize=(11, 14)); plt.imshow(img); plt.axis("off"); plt.show()

## 6. Robustness (O6) — detection completeness
The injection-recovery test (see `injection_recovery.ipynb`) quantifies the sensitivity floor: what fraction of injected transits the pipeline recovers vs SNR. This is the most credible evidence of pipeline quality and explains *which* planets in the sector are recoverable at a given signal level.

---
**Summary.** From one real TESS sector we (O2) blindly detected periodic events, (O3) characterised their shapes with MCMC, (O4) trained a calibrated classifier on the known TOIs, (O5) classified the unknown stars into transit / EB / blend / other, and (O6) reported parameters with significance plus an injection-recovery robustness curve — the full problem-statement workflow on real, noisy data.